In [11]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('✅ Imports loaded.')

✅ Imports loaded.


In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set the screen date (YYYY-MM-DD). Use 'today' to auto-set to today's date.
SCREEN_DATE = 'today'  # or e.g. '2025-03-24'

# Directory where your TradingView CSVs live.
# The screener will look for a file named:  tv_screen_gap-up_<SCREEN_DATE>.csv
TV_EXPORT_DIR = os.path.expanduser('~/Desktop/ALGOS/pmgus')

# ── Auto-resolve date ────────────────────────────────────────────────────────
if SCREEN_DATE.lower() == 'today':
    SCREEN_DATE = date.today().strftime('%Y-%m-%d')

TV_CSV_PATH = os.path.join(TV_EXPORT_DIR, f'tv_screen_gap-up_{SCREEN_DATE}.csv')
print(f'📅 Screen date  : {SCREEN_DATE}')
print(f'📂 CSV path     : {TV_CSV_PATH}')

📅 Screen date  : 2026-04-01
📂 CSV path     : /Users/sudz4/Desktop/ALGOS/pmgus/tv_screen_gap-up_2026-04-01.csv


In [3]:
# ── LOAD TRADINGVIEW CSV ──────────────────────────────────────────────────────
if not os.path.exists(TV_CSV_PATH):
    raise FileNotFoundError(
        f"\n❌ CSV not found at:\n   {TV_CSV_PATH}\n"
        f"Export your TradingView screener and save it to that path, "
        f"then re-run this cell."
    )

raw_df = pd.read_csv(TV_CSV_PATH)
print(f'✅ Loaded {len(raw_df)} rows from TradingView export.')
print(f'   Columns ({len(raw_df.columns)}): {list(raw_df.columns)}')

✅ Loaded 2065 rows from TradingView export.
   Columns (92): ['Symbol', 'Description', 'Industry', 'Sector', 'Exchange', 'Index', 'Market capitalization', 'Market capitalization - Currency', 'Price', 'Price - Currency', 'Pre-market Open', 'Pre-market Open - Currency', 'Pre-market Change', 'Pre-market Change - Currency', 'Pre-market Change %', 'Pre-market Gap %', 'Free float', 'Volume 1 day', 'Volume 1 week', 'Pre-market Volume', 'Average Volume 10 days', 'Average Volume 30 days', 'Average Volume 90 days', 'Volatility 1 day', 'Volatility 1 week', 'Volatility 1 month', 'Volume Weighted Average Price 1 day', 'Price to earnings ratio', 'Relative Volume at Time', 'Relative Volume 1 day', 'Beta 1 year', 'Beta 3 years', 'Beta 5 years', 'Relative Volume 1 minute', 'Relative Volume 5 minutes', 'Relative Volume 15 minutes', 'Relative Volume 30 minutes', 'Relative Volume 1 hour', 'Relative Volume 2 hours', 'Relative Volume 4 hours', 'Relative Volume 1 week', 'Relative Volume 1 month', 'High 1 mon

In [4]:
# ── MARKET CAP CATEGORIZATION ─────────────────────────────────────────────────
# Thresholds (USD).  Adjust freely — labels propagate through the whole screener.
MARKET_CAP_TIERS = [
    ('Titans',      200_000_000_000, float('inf')),
    ('Large caps',   10_000_000_000, 200_000_000_000),
    ('Mid caps',      2_000_000_000,  10_000_000_000),
    ('Small caps',      300_000_000,   2_000_000_000),
    ('Micro caps',       50_000_000,     300_000_000),
    ('Shrimp',                    0,      50_000_000),
]

def categorize_market_cap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Market capitalization'] = pd.to_numeric(df['Market capitalization'], errors='coerce')
    df['marketCapType'] = 'Undefined'
    for label, lo, hi in MARKET_CAP_TIERS:
        mask = (df['Market capitalization'] >= lo) & (df['Market capitalization'] < hi)
        df.loc[mask, 'marketCapType'] = label
    return df

setup_df = categorize_market_cap(raw_df)
setup_df = setup_df[setup_df['marketCapType'] != 'Undefined'].copy()

print('Market cap distribution in export:')
print(setup_df['marketCapType'].value_counts().to_string())

Market cap distribution in export:
marketCapType
Small caps    516
Mid caps      470
Large caps    449
Micro caps    303
Shrimp        272
Titans         48


In [5]:
# ── NUMERIC COERCION ──────────────────────────────────────────────────────────
NUMERIC_COLS = [
    'Market capitalization',
    'Float shares outstanding',
    'Price',
    'Pre-market Open',
    'Pre-market Change %',
    'Pre-market Gap %',
    'Pre-market Volume',
    'Relative Volume 1 day',
    'Relative Volume at Time',
    'Relative Volume 5 minutes',
    'Relative Volume 30 minutes',
    'Volume Weighted Average Price 1 day',
    'Volatility 1 day',
    'Volatility 1 week',
    'Volatility 1 month',
    'Average Volume 10 days',
    'Average Volume 30 days',
    'Average Volume 90 days',
    'Exponential Moving Average (3) 1 day',
    'Exponential Moving Average (7) 1 day',
]

for col in NUMERIC_COLS:
    if col in setup_df.columns:
        setup_df[col] = pd.to_numeric(setup_df[col], errors='coerce')

print(f'✅ Numeric coercion done. Working with {len(setup_df)} stocks.')

✅ Numeric coercion done. Working with 2058 stocks.


In [6]:
# ── SCREENING CRITERIA BY MARKET CAP TIER ────────────────────────────────────
# Each key maps directly to the marketCapType labels above.
# Tweak any value here — the filter logic below reads from this dict.
CRITERIA = {
    'Titans': {
        'min_pre_market_change_pct':  0.20,   # %
        'max_float_shares':           1_000_000_000,
        'min_rel_vol_1d':             1.20,
        'min_rel_vol_at_time':        0.03,
        'min_pre_market_gap_pct':     0.10,   # %
        'max_vwap_drawdown_pct':      0.30,   # price must be >= VWAP * (1 - x%)
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,   # vol_1d >= vol_1w >= vol_1m
    },
    'Large caps': {
        'min_pre_market_change_pct':  0.50,
        'max_float_shares':           200_000_000,
        'min_rel_vol_1d':             1.30,
        'min_rel_vol_at_time':        0.04,
        'min_pre_market_gap_pct':     0.50,
        'max_vwap_drawdown_pct':      0.40,
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,
    },
    'Mid caps': {
        'min_pre_market_change_pct':  2.00,
        'max_float_shares':           50_000_000,
        'min_rel_vol_1d':             1.30,
        'min_rel_vol_at_time':        0.05,
        'min_pre_market_gap_pct':     2.00,
        'max_vwap_drawdown_pct':      0.50,
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,
    },
    'Small caps': {
        'min_pre_market_change_pct':  3.00,
        'max_float_shares':           20_000_000,
        'min_rel_vol_1d':             1.20,
        'min_rel_vol_at_time':        0.05,
        'min_pre_market_gap_pct':     3.00,
        'max_vwap_drawdown_pct':      0.60,
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,
    },
    'Micro caps': {
        'min_pre_market_change_pct':  4.00,
        'max_float_shares':           5_000_000,
        'min_rel_vol_1d':             1.10,
        'min_rel_vol_at_time':        0.05,
        'min_pre_market_gap_pct':     4.00,
        'max_vwap_drawdown_pct':      0.70,
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,
    },
    'Shrimp': {
        'min_pre_market_change_pct':  5.00,
        'max_float_shares':           1_000_000,
        'min_rel_vol_1d':             1.00,
        'min_rel_vol_at_time':        0.05,
        'min_pre_market_gap_pct':     5.00,
        'max_vwap_drawdown_pct':      0.80,
        'min_pre_market_volume':      50_000,
        'require_volatility_trend':   True,
    },
}

print('✅ Criteria config ready.')

✅ Criteria config ready.


In [7]:
# ── FILTER ENGINE ─────────────────────────────────────────────────────────────
def _col(df, name):
    """Return column as Series, or NaN series if column is missing."""
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)


def apply_criteria(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Apply a criteria dict to a slice of the DataFrame and return matches."""
    mask = pd.Series(True, index=df.index)

    # Pre-market change %
    mask &= _col(df, 'Pre-market Change %') >= cfg['min_pre_market_change_pct']

    # Pre-market gap %
    mask &= _col(df, 'Pre-market Gap %') >= cfg['min_pre_market_gap_pct']

    # Float shares
    float_col = _col(df, 'Float shares outstanding')
    mask &= float_col.isna() | (float_col <= cfg['max_float_shares'])

    # Relative volume — 1 day
    mask &= _col(df, 'Relative Volume 1 day') >= cfg['min_rel_vol_1d']

    # Relative volume at time
    mask &= _col(df, 'Relative Volume at Time') >= cfg['min_rel_vol_at_time']

    # Pre-market volume
    pm_vol = _col(df, 'Pre-market Volume')
    if not pm_vol.isna().all():
        mask &= pm_vol >= cfg['min_pre_market_volume']

    # VWAP drawdown: price >= VWAP * (1 - drawdown_pct/100)
    vwap = _col(df, 'Volume Weighted Average Price 1 day')
    price = _col(df, 'Price')
    if not vwap.isna().all():
        drawdown_factor = 1 - cfg['max_vwap_drawdown_pct'] / 100
        mask &= price >= vwap * drawdown_factor

    # Volatility trend: vol_1d >= vol_1w AND vol_1d >= vol_1m
    if cfg.get('require_volatility_trend', False):
        v1d = _col(df, 'Volatility 1 day')
        v1w = _col(df, 'Volatility 1 week')
        v1m = _col(df, 'Volatility 1 month')
        if not (v1d.isna().all() or v1w.isna().all()):
            mask &= v1d >= v1w
        if not (v1d.isna().all() or v1m.isna().all()):
            mask &= v1d >= v1m

    return df[mask]


def run_screener(df: pd.DataFrame, criteria: dict) -> pd.DataFrame:
    """Loop over each market cap tier, apply criteria, return combined results."""
    results = []
    for tier, cfg in criteria.items():
        tier_df = df[df['marketCapType'] == tier]
        if tier_df.empty:
            continue
        matched = apply_criteria(tier_df, cfg)
        if not matched.empty:
            results.append(matched)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


print('✅ Filter engine defined.')

✅ Filter engine defined.


In [12]:
# ── RUN SCREENER ──────────────────────────────────────────────────────────────
OUTPUT_COLS = [
    'Symbol',
    'Description',
    'marketCapType',
    'Pre-market Change %',
    'Pre-market Gap %',
    'Market capitalization',
    'Price',
    'Pre-market Open',
    'Industry',
    'Sector',
    'Exchange',
    'Index',
    'Recent earnings date',
    'Upcoming earnings date',
    'Float shares outstanding',
    'Pre-market Volume',
    'Average Volume 10 days',
    'Average Volume 30 days',
    'Average Volume 90 days',
    'Relative Volume 1 day',
    'Relative Volume 5 minutes',
    'Relative Volume 30 minutes',
    'Relative Volume at Time',
    'Volume Weighted Average Price 1 day',
    'Volatility 1 day',
    'Volatility 1 week',
    'Volatility 1 month',
    'Analyst Rating',
    'Technical Rating 5 minutes',
    'Exponential Moving Average (3) 1 day',
    'Exponential Moving Average (7) 1 day',
]

screened_df = run_screener(setup_df, CRITERIA)

# Keep only columns that are actually present in the data
present_cols = [c for c in OUTPUT_COLS if c in screened_df.columns]
screened_df = screened_df[present_cols]

# Sort: highest pre-market change first, then by price descending
screened_df = screened_df.sort_values(
    by=['Pre-market Change %', 'Price'],
    ascending=[False, False]
).reset_index(drop=True)

print(f'📅 Screen date  : {SCREEN_DATE}')
print(f'📥 Raw export   : {len(raw_df)} stocks')
print(f'🎯 After screen : {len(screened_df)} stocks')
display(screened_df)

📅 Screen date  : 2026-04-01
📥 Raw export   : 2065 stocks
🎯 After screen : 14 stocks


,Symbol,Description,marketCapType,Pre-market Change %,Pre-market Gap %,Market capitalization,Price,Pre-market Open,Industry,Sector,Exchange,Index,Recent earnings date,Upcoming earnings date,Pre-market Volume,Average Volume 10 days,Average Volume 30 days,Average Volume 90 days,Relative Volume 1 day,Relative Volume 5 minutes,Relative Volume 30 minutes,Relative Volume at Time,Volume Weighted Average Price 1 day,Volatility 1 day,Volatility 1 week,Volatility 1 month,Analyst Rating,Technical Rating 5 minutes,Exponential Moving Average (3) 1 day,Exponential Moving Average (7) 1 day
0,ELAB,PMGC Holdings Inc.,Shrimp,16.69,29.38,"7,580,454.00",14.00,7.75,Biotechnology,Health technology,NASDAQ,"NASDAQ Composite, NASDAQ Industrials",NaN,NaN,3620981,"25,959,938.40","8,752,256.61","3,041,762.79",2.01,0.80,1.67,2.43,11.52,122.12,57.58,26.75,No rating,Strong buy,9.19,6.23
1,SHAZ,"SharonAI Holdings, Inc.",Small caps,14.39,3.21,"441,887,684.60",27.62,23.46,Data processing services,Technology services,NASDAQ,"NASDAQ Composite, NASDAQ Computer",NaN,NaN,112286,"187,722.20","177,024.37","64,719.14",5.44,10.35,2.07,5.51,26.35,28.13,12.22,19.99,No rating,Strong buy,25.14,24.22
2,WDC,Western Digital Corporation,Large caps,3.83,2.81,"100,941,783,740.00",297.73,278.10,Computer peripherals,Electronic technology,NASDAQ,"S&P 500, NASDAQ 100, NASDAQ Composite, S&P 500 Information Technology, S&P 500 ESG, Nasdaq US Large Cap Growth, NASDAQ Computer, NASDAQ-100 Technology Sector, Russell 3000, Russell 1000, STOXX Global 1800, STOXX North America 600, STOXX USA 500, STOXX Global 1800 ex Asia/Pacific, STOXX Global 1800 ex Europe, STOXX Developed Markets 2400, STOXX USA 500",2026-01-29,2026-04-23,81125,"9,323,597.90","9,245,403.17","9,240,015.64",1.39,5.58,1.84,1.56,294.39,12.29,8.40,7.92,Buy,Sell,283.00,281.47
3,MU,"Micron Technology, Inc.",Titans,3.28,2.68,"414,835,480,500.00",367.85,346.90,Semiconductors,Electronic technology,NASDAQ,"S&P 500, NASDAQ 100, NASDAQ Composite, PHLX Semiconductor Sector, S&P 500 Information Technology, S&P 500 ESG, Nasdaq US Large Cap Growth, NASDAQ Computer, NASDAQ-100 Technology Sector, Russell 3000, Russell 1000, STOXX Global 1800, STOXX North America 600, STOXX USA 500, STOXX Global 1800 ex Asia/Pacific, STOXX Global 1800 ex Europe, STOXX Global 200, STOXX Developed Markets 2400, STOXX USA 500",2026-03-18,2026-07-01,2026591,"61,595,583.60","43,311,679.77","36,265,586.39",1.24,2.47,0.92,1.37,362.91,11.68,8.73,6.71,Strong buy,Sell,354.39,363.62
4,ING,"ING Group, N.V.",Large caps,3.11,1.57,"72,397,952,831.71",26.81,26.46,Major banks,Finance,NYSE,NaN,2026-01-29,2026-04-30,316357,"4,236,445.20","3,383,735.30","2,633,907.89",1.35,3.18,2.01,1.46,26.83,3.77,2.13,2.63,Buy,Sell,26.15,25.77
5,UBS,UBS Group AG Registered,Large caps,2.94,2.00,"122,864,057,735.98",39.74,39.85,Investment managers,Finance,NYSE,NaN,2026-02-04,2026-04-29,155236,"3,287,631.10","2,995,749.43","2,516,530.50",1.47,8.44,2.27,1.55,39.84,3.02,2.08,2.30,Neutral,Buy,38.84,38.12
6,HSBC,"HSBC Holdings, plc.",Titans,2.85,0.58,"277,506,574,790.18",85.46,82.97,Major banks,Finance,NYSE,NaN,2026-02-24,2026-05-05,53293,"2,669,722.10","2,356,450.10","2,088,074.71",1.80,8.01,2.31,2.02,85.34,4.42,2.08,2.36,Buy,Buy,83.22,81.67
7,STLA,Stellantis N.V.,Large caps,2.54,1.55,"21,481,964,474.43",7.43,7.20,Motor vehicles,Consumer durables,NYSE,NaN,2026-02-26,2026-07-30,438167,"26,585,403.40","21,119,940.10","16,785,273.78",1.50,5.72,1.88,1.48,7.35,4.87,3.33,3.59,Buy,Buy,7.17,6.97
8,BA,Boeing Company (The),Large caps,2.12,0.68,"162,818,140,040.00",207.32,200.39,Aerospace & defense,Electronic technology,NYSE,"S&P 500, Dow Jones Industrial Average, S&P 500 Industrials, Nasdaq US Large Cap Growth, S&P 100, Russell 3000, Dow Jones Composite Average, Russell 1000, STOXX Global 1800, STOXX North America 600, STOXX USA 500, STOXX Global 1800 ex Asia/Pacific, STOXX Global 1800 ex Europe, STOXX Americas 100, STOXX Global 200, STOXX Developed Markets 2400, STOXX USA 500",2026-01-27,2026-04-29,76127,"

In [13]:
# ── SECTOR BREAKDOWN ──────────────────────────────────────────────────────────
if 'Sector' in screened_df.columns:
    sector_df = (
        screened_df.groupby('Sector')
        .size()
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
        .reset_index(drop=True)
    )
    print('Sector breakdown:')
    display(sector_df)

Sector breakdown:


,Sector,Count
0,Electronic technology,5
1,Finance,3
2,Health technology,2
3,Consumer durables,1
4,Energy minerals,1
5,Non-energy minerals,1
6,Technology services,1


In [14]:
# ── MARKET CAP TIER BREAKDOWN ─────────────────────────────────────────────────
tier_order = ['Titans', 'Large caps', 'Mid caps', 'Small caps', 'Micro caps', 'Shrimp']
tier_df = (
    screened_df.groupby('marketCapType')
    .size()
    .reindex([t for t in tier_order if t in screened_df['marketCapType'].unique()])
    .reset_index(name='Count')
    .rename(columns={'marketCapType': 'Tier'})
)
print('Market cap tier breakdown:')
display(tier_df)

Market cap tier breakdown:


,Tier,Count
0,Titans,4
1,Large caps,8
2,Small caps,1
3,Shrimp,1


In [15]:
# ── FINAL SYMBOL LIST ─────────────────────────────────────────────────────────
symbol_list = screened_df['Symbol'].tolist()
print(f'Total symbols ({len(symbol_list)}):')
print(symbol_list)

Total symbols (14):
['ELAB', 'SHAZ', 'WDC', 'MU', 'ING', 'UBS', 'HSBC', 'STLA', 'BA', 'MRVL', 'INTC', 'AA', 'GSK', 'SHEL']


In [16]:
# ── EMA MOMENTUM FILTER  (EMA3 > EMA7 = short-term uptrend) ──────────────────
ema3_col = 'Exponential Moving Average (3) 1 day'
ema7_col = 'Exponential Moving Average (7) 1 day'

if ema3_col in screened_df.columns and ema7_col in screened_df.columns:
    ema_mask = screened_df[ema3_col] > screened_df[ema7_col]
    ema_df = screened_df[ema_mask].copy()
    print(f'EMA3 > EMA7 (short-term uptrend): {len(ema_df)} stocks')
    print(ema_df['Symbol'].tolist())
    display(ema_df[['Symbol', 'Description', 'marketCapType',
                     'Pre-market Change %', 'Pre-market Gap %',
                     ema3_col, ema7_col]])
else:
    print('EMA columns not found in data — skipping EMA filter.')

EMA3 > EMA7 (short-term uptrend): 13 stocks
['ELAB', 'SHAZ', 'WDC', 'ING', 'UBS', 'HSBC', 'STLA', 'BA', 'MRVL', 'INTC', 'AA', 'GSK', 'SHEL']


,Symbol,Description,marketCapType,Pre-market Change %,Pre-market Gap %,Exponential Moving Average (3) 1 day,Exponential Moving Average (7) 1 day
0,ELAB,PMGC Holdings Inc.,Shrimp,16.69,29.38,9.19,6.23
1,SHAZ,"SharonAI Holdings, Inc.",Small caps,14.39,3.21,25.14,24.22
2,WDC,Western Digital Corporation,Large caps,3.83,2.81,283.00,281.47
4,ING,"ING Group, N.V.",Large caps,3.11,1.57,26.15,25.77
5,UBS,UBS Group AG Registered,Large caps,2.94,2.00,38.84,38.12
6,HSBC,"HSBC Holdings, plc.",Titans,2.85,0.58,83.22,81.67
7,STLA,Stellantis N.V.,Large caps,2.54,1.55,7.17,6.97
8,BA,Boeing Company (The),Large caps,2.12,0.68,201.26,199.03
9,MRVL,"Marvell Technology, Inc.",Large caps,2.11,2.88,101.03,97.17
10,INTC,Intel Corporation,Titans,1.93,0.84,45.70,44.86


In [17]:
# ── ANALYST + TECHNICAL RATING FILTER ────────────────────────────────────────
# Keep stocks where BOTH analyst rating AND 5-min technical rating are bullish.
BULLISH_RATINGS = {'Buy', 'Strong buy'}

analyst_col = 'Analyst Rating'
tech_col    = 'Technical Rating 5 minutes'

if analyst_col in screened_df.columns and tech_col in screened_df.columns:
    rating_mask = (
        screened_df[analyst_col].isin(BULLISH_RATINGS) &
        screened_df[tech_col].isin(BULLISH_RATINGS)
    )
    rated_df = screened_df[rating_mask].copy()
    print(f'Bullish on BOTH analyst + 5-min technical rating: {len(rated_df)} stocks')
    display(rated_df[['Symbol', 'Description', 'marketCapType',
                       'Pre-market Change %', 'Pre-market Gap %',
                       analyst_col, tech_col]])
else:
    print('Rating columns not present — skipping rating filter.')

Bullish on BOTH analyst + 5-min technical rating: 5 stocks


,Symbol,Description,marketCapType,Pre-market Change %,Pre-market Gap %,Analyst Rating,Technical Rating 5 minutes
6,HSBC,"HSBC Holdings, plc.",Titans,2.85,0.58,Buy,Buy
7,STLA,Stellantis N.V.,Large caps,2.54,1.55,Buy,Buy
9,MRVL,"Marvell Technology, Inc.",Large caps,2.11,2.88,Strong buy,Buy
11,AA,Alcoa Corporation,Large caps,1.49,1.09,Buy,Strong buy
13,SHEL,Shell PLC,Titans,0.26,0.28,Buy,Buy


In [18]:
# ── HIGH-CONVICTION SHORTLIST ─────────────────────────────────────────────────
# Intersection: passed screener + EMA uptrend + bullish ratings
# Adjust the component DataFrames (ema_df, rated_df) as needed.

high_conviction_syms = (
    set(ema_df['Symbol'].tolist()) &
    set(rated_df['Symbol'].tolist())
)

hc_df = screened_df[screened_df['Symbol'].isin(high_conviction_syms)].copy()

print(f'\n🔥 HIGH-CONVICTION SHORTLIST ({len(hc_df)} stocks):')
print(hc_df['Symbol'].tolist())
display(hc_df[['Symbol', 'Description', 'marketCapType',
               'Pre-market Change %', 'Pre-market Gap %', 'Price',
               'Relative Volume 1 day', 'Analyst Rating', 'Technical Rating 5 minutes']])


🔥 HIGH-CONVICTION SHORTLIST (5 stocks):
['HSBC', 'STLA', 'MRVL', 'AA', 'SHEL']


,Symbol,Description,marketCapType,Pre-market Change %,Pre-market Gap %,Price,Relative Volume 1 day,Analyst Rating,Technical Rating 5 minutes
6,HSBC,"HSBC Holdings, plc.",Titans,2.85,0.58,85.46,1.80,Buy,Buy
7,STLA,Stellantis N.V.,Large caps,2.54,1.55,7.43,1.50,Buy,Buy
9,MRVL,"Marvell Technology, Inc.",Large caps,2.11,2.88,106.71,2.00,Strong buy,Buy
11,AA,Alcoa Corporation,Large caps,1.49,1.09,72.06,1.58,Buy,Strong buy
13,SHEL,Shell PLC,Titans,0.26,0.28,92.03,1.70,Buy,Buy
